# Week 4 - Frozen segmentation heads

Run the pseudo-label notebook first. Use a Colab GPU and store the clone on persistent storage if you want to keep feature caches and checkpoints.

In [ ]:
%pip install 'numpy<2' 'scipy<1.16' 'astropy<7' 'photutils==2.3.0' 'pandas<3' 'pyarrow<22' huggingface_hub

In [ ]:
# Importing necessary libraries
import json
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Week4").exists())
W4, W5 = REPO / "Week4", REPO / "Week5"
sys.path[:0] = [str(W4 / "src"), str(W5 / "src")]


In [ ]:
# Setting the device and seed
import torch
from torch.utils.data import DataLoader
import pandas as pd
from heads import SETUPS, Masks, backbone, train_head, load_head, predict, naive_attention
from metrics import confusion, iou_summary, binary_iou

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print("Device:", DEVICE)


In [ ]:
# Loading the saved split
splits = json.loads((W4 / "data/splits.json").read_text())
def real_records(ids):
    excluded = set()
    for filename in ("automatic_exclusions.json", "excluded_ids.json"):
        path = W4 / "data" / filename
        if path.exists():
            excluded.update(json.loads(path.read_text()))
    ids = [key for key in ids if key not in excluded]
    return [(key, W4 / f"data/cutouts/{key}.npz", W4 / f"results/pseudo_labels/{key}.npy")
            for key in ids]


In [ ]:
# Training the Linear heads and +ms
checkpoints = W4 / "results/checkpoints"
metrics = W4 / "results/metrics"
checkpoints.mkdir(parents=True, exist_ok=True)
metrics.mkdir(parents=True, exist_ok=True)

for name, setup in SETUPS.items():
    batch_size = 32 if setup[1] == 224 else 16
    train = DataLoader(Masks(real_records(splits["train"])), batch_size=batch_size, shuffle=True)
    val = DataLoader(Masks(real_records(splits["validation"])), batch_size=batch_size)
    history = train_head(setup, train, val, DEVICE, checkpoints / f"{name}.pt",
                         confusion, feature_cache=W4 / "results/features/linear_224"
                         if name == "linear_224" else None)
    pd.DataFrame(history).to_csv(metrics / f"{name}_learning.csv", index=False)


In [ ]:
# Evaluating the selected checkpoints on test
test = DataLoader(Masks(real_records(splits["test"])), batch_size=4)
models = {name: load_head(checkpoints / f"{name}.pt", DEVICE) for name in SETUPS}
dino = backbone("dino_vits8", DEVICE)
pred_dir = W4 / "results/predictions"
pred_dir.mkdir(parents=True, exist_ok=True)
counts = {name: np.zeros((3, 3), dtype=np.int64) for name in models}
counts.update({name: np.zeros((3, 3), dtype=np.int64) for name in ("ms_no_tta",)})
binary = {name: np.zeros((2, 2), dtype=np.int64) for name in ("target", "all_sources")}
hard_ids = json.loads((W4 / "data/hard_set.json").read_text())
assert set(hard_ids) <= set(splits["test"])
gallery = {}

for keys, images, labels in test:
    labels = labels.numpy()
    preds = {name: predict(head, model, images, setup, DEVICE, tta=name == "ms")
             for name, (head, model, setup) in models.items()}
    preds["ms_no_tta"] = predict(*models["ms"][:2], images, models["ms"][2], DEVICE)
    naive = naive_attention(dino, images, DEVICE)
    for name, pred in preds.items():
        counts[name] += confusion(labels, pred)
    binary["target"] += confusion(labels == 1, naive, classes=2)
    binary["all_sources"] += confusion(labels > 0, naive, classes=2)
    for i, key in enumerate(keys):
        np.savez_compressed(pred_dir / f"{key}.npz", naive=naive[i],
                            **{name: pred[i] for name, pred in preds.items()})
        if key in hard_ids:
            gallery[key] = (images[i].permute(1, 2, 0), labels[i], naive[i],
                            preds["linear_224"][i], preds["ms"][i])

rows = [{"method": "naive", "backbone": "dino_vits8", "input": 224, "grid": 28,
         "target_iou": iou_summary(binary["target"])["target_iou"],
         "all_sources_iou": iou_summary(binary["all_sources"])["target_iou"]}]
for name, cm in counts.items():
    model, size, _ = SETUPS["ms" if name == "ms_no_tta" else name]
    rows.append({"method": name, "backbone": model, "input": size,
                 "grid": size // (8 if model == "dino_vits8" else 14), **iou_summary(cm)})
table = pd.DataFrame(rows)
table.to_csv(metrics / "comparison.csv", index=False)
display(table)


In [ ]:
# Visualizing the hard set
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["black", "gold", "royalblue"])
for start in range(0, len(hard_ids), 5):
    keys = hard_ids[start:start + 5]
    fig, axes = plt.subplots(len(keys), 5, figsize=(13, 2.5 * len(keys)), squeeze=False)
    for key, row in zip(keys, axes):
        titles = (key, "Pseudo-label", "Naive mask", "Linear 224", "+ms")
        for col, (ax, panel, title) in enumerate(zip(row, gallery[key], titles)):
            ax.imshow(panel, cmap="gray" if col == 2 else cmap,
                      vmin=0, vmax=1 if col == 2 else 2)
            ax.set_title(title)
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(pred_dir / f"hard_set_{start // 5 + 1}.png", dpi=130)
    plt.show()
